# Buzzer with Square Wave

In [12]:
import threading
import time
import socket

from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
btns = base.btns_gpio

In [13]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
//make sure to have it return something
int write_gpio(unsigned int pin, unsigned int val){ 
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}
int clear_gpios(){
    for(int i = 0; i < 8; ++i){
        write_gpio(i,0);
    }
    return 0;
}

In [14]:
buzz_pin=0

def alarm_tone(tone_freq):
    
    sleep_time=1/tone_freq
    print(f"sleep time: {sleep_time}")
    start_time = time.time()
    
    while time.time() < start_time+0.5:
        write_gpio(buzz_pin, 1)
        time.sleep(sleep_time)
        write_gpio(buzz_pin, 0)
        time.sleep(sleep_time)
        
    write_gpio(buzz_pin, 0)
    print("alarm stop")

def wait_btn_client_start():
    while True:
        if not btns[0].read():
            time.sleep(0.1)
        else: break

In [ ]:
#only for test
clear_gpios()
alarm_tone(3000)

# while True:
#     if btns[0].read != 0:
#         alarm_tone(1500)
#     elif btns[1].read != 0:
#         alarm_tone(2500)
#     elif btns[2].read != 0:
#         alarm_tone(3500)
#     elif btns[3].read != 0:
#         break
#     else: time.sleep(0.5)
    

# Server

In [15]:
def server_t(port_ID):
    print(f"inside server thread")
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    print(f"binding socket")
    sock.bind(('0.0.0.0', port_ID))
    
    print(f"server starts listening via {port_ID}")
    sock.listen(1)

    conn, addr = sock.accept()
    print(f"Accepted connection from {port_ID}")

    while True:
        data = conn.recv(1024)
        if not data:
            print("Client disconneted")
            break
        if data:
            print("Received message: ", data.decode('utf8'))
            alarm_tone(3000)

# Client

In [16]:
def client_t(port_ID, IP_addr):
    message = "buzzzzz"
    print(f"inside client thread")
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    print("connect client")
    sock.connect((IP_addr, port_ID))
    
    print("entering while loop")
    while True:
        if btns[1].read() != 0:
            time.sleep(0.5)
            sock.sendall(message.encode('utf-8'))
        elif btns[2].read() != 0:
            time.sleep(0.1)
            print("client disconnect button")
            break
        else: pass
            
    # Close the socket
    print(f"closing the socket")
    sock.close()

# All combined

In [10]:
server_port = 9999
client_port = 8888
ip_addr = "192.168.0.170"

clear_gpios()

threads = []

t = threading.Thread(target=server_t, args=(server_port,))
threads.append(t)
t.start()
print("starting server thread")

t = threading.Thread(target=client_t, args=(client_port, ip_addr))
threads.append(t)
wait_btn_client_start()
t.start()
print("starting client thread")

for t in threads:
    name = t.name
    t.join()
    print('{} joined'.format(name))

inside server thread
starting server thread
binding socket
server starts listening via 9999
Accepted connection from 9999
inside client thread
connect client
starting client thread
entering while loop
Received message:  BEEP
sleep time: 0.0003333333333333333
alarm stop
Received message:  BEEP
sleep time: 0.0003333333333333333
alarm stop
Received message:  BEEP
sleep time: 0.0003333333333333333
alarm stop
Received message:  BEEP
sleep time: 0.0003333333333333333
alarm stop
Received message:  BEEP
sleep time: 0.0003333333333333333
alarm stop
Received message:  BEEP
sleep time: 0.0003333333333333333
alarm stop
Client disconneted
Thread-11 (server_t) joined
client disconnect button
closing the socket
Thread-12 (client_t) joined
